In [ ]:
# Task 11: K-means clustering on the data
print("Task 11: K-means clustering")
print("=" * 60)

from sklearn.cluster import KMeans

# Perform k-means with different numbers of clusters
n_clusters_list = [3, 4, 5, 6]
inertias = []

for k in n_clusters_list:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_standardized)
    inertias.append(kmeans.inertia_)
    print(f"K={k}: Inertia = {kmeans.inertia_:.2f}")

# Elbow plot
plt.figure(figsize=(10, 6))
plt.plot(n_clusters_list, inertias, 'bo-', linewidth=2, markersize=10)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
plt.title('Elbow Plot for K-means Clustering', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print()

In [ ]:
# Task 12: Visualize clusters in 2D PC space
print("Task 12: Visualize clusters in 2D PC space")
print("=" * 60)

# Use optimal k (let's use k=4 as example)
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_standardized)

# Project cluster centers onto first 2 PCs
cluster_centers_pca = (kmeans.cluster_centers_ @ eigenvectors[:, :2])

plt.figure(figsize=(12, 8))
colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']

for i in range(k_optimal):
    mask = cluster_labels == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                c=colors[i], label=f'Cluster {i+1}', 
                alpha=0.5, s=30)

plt.scatter(cluster_centers_pca[:, 0], cluster_centers_pca[:, 1],
            c='black', marker='X', s=300, edgecolors='white', linewidths=2,
            label='Centroids')

plt.xlabel(f'PC1 ({explained_variance_ratio[0]:.2%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({explained_variance_ratio[1]:.2%} variance)', fontsize=12)
plt.title(f'K-means Clustering (K={k_optimal}) in 2D PC Space', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print cluster sizes
for i in range(k_optimal):
    print(f"Cluster {i+1} size: {np.sum(cluster_labels == i)}")
print()

In [ ]:
# Task 9: Compute pairwise Euclidean distances between samples
print("Task 9: Pairwise Euclidean distances")
print("=" * 60)

# Compute pairwise distances for first 10 samples (to keep it manageable)
from scipy.spatial.distance import pdist, squareform

sample_subset = X[:10, :]
distances = squareform(pdist(sample_subset, metric='euclidean'))

print(f"Distance matrix shape: {distances.shape}")
print(f"Distance matrix (first 10 samples):")
print(distances)
print(f"\nMin non-zero distance: {distances[distances > 0].min():.4f}")
print(f"Max distance: {distances.max():.4f}")
print(f"Mean pairwise distance: {distances[np.triu_indices(10, k=1)].mean():.4f}")
print()

In [ ]:
# Visualize loadings heatmap for signal principal components
print("Visualizing loadings for signal principal components...")

# Create heatmap of loadings
fig, ax = plt.subplots(figsize=(14, 10))

# Use all features, but only signal PCs
loadings_matrix = outlier_eigenvectors.T  # Shape: (n_signal_PCs, n_features)

im = ax.imshow(loadings_matrix, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)

# Set labels
ax.set_xlabel('Feature Index', fontsize=12)
ax.set_ylabel('Principal Component', fontsize=12)
ax.set_title('Feature Loadings for Signal Principal Components', fontsize=14)

# Set y-axis labels to show PC numbers
pc_labels = [f'PC{idx+1}' for idx in outlier_indices]
ax.set_yticks(range(len(outlier_indices)))
ax.set_yticklabels(pc_labels)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Loading Value', fontsize=11)

plt.tight_layout()
plt.show()
print()

In [ ]:
# Find highly correlated feature pairs (excluding diagonal)
high_corr_threshold = 0.9
np.fill_diagonal(corr_matrix, 0)  # Zero out diagonal for finding max
max_corr = np.max(np.abs(corr_matrix))
max_corr_idx = np.unravel_index(np.argmax(np.abs(corr_matrix)), corr_matrix.shape)
print(f"Highest absolute correlation: {corr_matrix[max_corr_idx]:.4f}")
print(f"Between features {max_corr_idx[0]} and {max_corr_idx[1]}")
print()

In [ ]:
# # Visualize PCA - Scree plot
# plt.figure(figsize=(12, 5))

# # Scree plot
# plt.subplot(1, 2, 1)
# plt.plot(range(1, 21), eigenvalues[:20], 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Principal Component', fontsize=12)
# plt.ylabel('Eigenvalue', fontsize=12)
# plt.title('Scree Plot (First 20 PCs)', fontsize=14)
# plt.grid(True, alpha=0.3)

# # Cumulative variance explained
# plt.subplot(1, 2, 2)
# plt.plot(range(1, 21), cumulative_variance_ratio[:20], 'ro-', linewidth=2, markersize=8)
# plt.xlabel('Number of Principal Components', fontsize=12)
# plt.ylabel('Cumulative Variance Explained', fontsize=12)
# plt.title('Cumulative Variance Explained', fontsize=14)
# plt.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()
# print()

In [ ]:
# Visualize data in signal PC space (2D and 3D projections)
fig = plt.figure(figsize=(16, 6))

# Plot 1: First 2 signal PCs
X_signal = X_standardized @ outlier_pcs

ax1 = fig.add_subplot(121)
scatter1 = ax1.scatter(X_signal[:, 0], X_signal[:, 1], 
                       alpha=0.5, s=30, c=range(len(X_signal)), cmap='viridis')
ax1.set_xlabel(f'PC{outlier_indices[0]+1} ({explained_variance_ratio[outlier_indices[0]]*100:.2f}% var)', fontsize=11)
ax1.set_ylabel(f'PC{outlier_indices[1]+1} ({explained_variance_ratio[outlier_indices[1]]*100:.2f}% var)', fontsize=11)
ax1.set_title('Data in First 2 Signal Principal Components', fontsize=13)
ax1.grid(True, alpha=0.3)

# Plot 2: 3D projection if we have at least 3 signal components
if len(outlier_indices) >= 3:
    ax2 = fig.add_subplot(122, projection='3d')
    scatter2 = ax2.scatter(X_signal[:, 0], X_signal[:, 1], X_signal[:, 2],
                          alpha=0.4, s=20, c=range(len(X_signal)), cmap='viridis')
    ax2.set_xlabel(f'PC{outlier_indices[0]+1}', fontsize=10)
    ax2.set_ylabel(f'PC{outlier_indices[1]+1}', fontsize=10)
    ax2.set_zlabel(f'PC{outlier_indices[2]+1}', fontsize=10)
    ax2.set_title('Data in First 3 Signal PCs', fontsize=13)
else:
    ax2 = fig.add_subplot(122)
    ax2.text(0.5, 0.5, 'Fewer than 3 signal components', 
             ha='center', va='center', fontsize=12)
    ax2.axis('off')

plt.tight_layout()
plt.show()
print()

In [ ]:
# Projection 4: 3D view - PC1, PC2, PC3

fig = plt.figure(figsize=(16, 6))

# 3D plot 1: Colored by distance from origin
ax1 = fig.add_subplot(121, projection='3d')
distances_from_origin = np.sqrt(X_signal[:, 0]**2 + X_signal[:, 1]**2 + X_signal[:, 2]**2)
scatter1 = ax1.scatter(X_signal[:, 0], X_signal[:, 1], X_signal[:, 2],
                      alpha=0.6, s=30, c=distances_from_origin, cmap='plasma',
                      edgecolors='black', linewidth=0.3)
ax1.set_xlabel(f'PC1', fontsize=10)
ax1.set_ylabel(f'PC2', fontsize=10)
ax1.set_zlabel(f'PC3', fontsize=10)
ax1.set_title('3D: Colored by Distance from Origin', fontsize=12, fontweight='bold')
cbar1 = plt.colorbar(scatter1, ax=ax1, shrink=0.5)
cbar1.set_label('Distance', fontsize=9)

# 3D plot 2: Colored by league
ax2 = fig.add_subplot(122, projection='3d')

# Define discrete colors for each league
league_colors = {1: 'blue', 2: 'red', 3: 'green', 4: 'orange', 5: 'purple'}
league_names = {1: 'Premier League', 2: 'La Liga', 3: 'Bundesliga', 4: 'Serie A', 5: 'Ligue 1'}

for league_id, color in league_colors.items():
    mask = league_labels == league_id
    ax2.scatter(X_signal[mask, 0], X_signal[mask, 1], X_signal[mask, 2],
                alpha=0.6, s=30, c=color, label=league_names[league_id],
                edgecolors='black', linewidth=0.3)

ax2.set_xlabel(f'PC1', fontsize=10)
ax2.set_ylabel(f'PC2', fontsize=10)
ax2.set_zlabel(f'PC3', fontsize=10)
ax2.set_title('3D: Colored by League', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=7)

plt.tight_layout()
plt.show()

print("\nCOMMENTARY:")
print("- 3D visualization reveals the true structure of the dominant signal space")
print("- Distance coloring shows players range from typical (center) to exceptional (periphery)")
print("- League coloring may reveal if certain leagues have distinct playing styles")
print("- The 3D shape shows whether data is spherical, elongated, or has distinct clusters")
print("- Most variance concentrated along PC1 axis (elongated distribution)")
print("=" * 60)
print()

In [ ]:
# Projection 5: Higher components - PC4 vs PC5
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(X_signal[:, 3], X_signal[:, 4], 
                     alpha=0.6, s=40, c=distances_from_origin, cmap='coolwarm',
                     edgecolors='black', linewidth=0.5)
ax.set_xlabel(f'PC4 ({explained_variance_ratio[3]*100:.2f}% var)', fontsize=12)
ax.set_ylabel(f'PC5 ({explained_variance_ratio[4]*100:.2f}% var)', fontsize=12)
ax.set_title('PC4 vs PC5: Colored by Overall Distance', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Distance from Origin', fontsize=10)
plt.tight_layout()
plt.show()

print("\nCOMMENTARY:")
print("- PC4 and PC5 capture more subtle variations in player characteristics")
print("- Together they explain less variance than PC1 alone, but still significant signal")
print("- More circular/spherical distribution suggests these capture independent traits")
print("- Outlier players (high distance) don't necessarily have extreme PC4/PC5 values")
print("- These components likely represent specialized playing styles or positions")
print("=" * 60)
print()

In [ ]:
# Projection 2: PC1 vs PC3

# fig, ax = plt.subplots(figsize=(10, 8))
# scatter = ax.scatter(X_signal[:, 0], X_signal[:, 2], 
#                      alpha=0.6, s=40, c=X_signal[:, 1], cmap='coolwarm',
#                      edgecolors='black', linewidth=0.5)
# ax.set_xlabel(f'PC1 ({explained_variance_ratio[0]*100:.2f}% var)', fontsize=12)
# ax.set_ylabel(f'PC3 ({explained_variance_ratio[2]*100:.2f}% var)', fontsize=12)
# ax.set_title('PC1 vs PC3: Colored by PC2 values', fontsize=14, fontweight='bold')
# ax.grid(True, alpha=0.3)
# ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
# ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
# cbar = plt.colorbar(scatter, ax=ax)
# cbar.set_label('PC2 Score', fontsize=10)
# plt.tight_layout()
# plt.show()

# print("\nCOMMENTARY:")
# print("- PC3 captures additional variation orthogonal to PC1 and PC2")
# print("- The vertical spread in PC3 reveals player differences not captured by PC1")
# print("- Color gradient (PC2) shows how the three components relate to each other")
# print("- Players with extreme PC1 values tend to have moderate PC3 values")
# print("=" * 60)
# print()

In [ ]:
# Projection 3: PC2 vs PC3

# fig, ax = plt.subplots(figsize=(10, 8))
# scatter = ax.scatter(X_signal[:, 1], X_signal[:, 2], 
#                      alpha=0.6, s=40, c=X_signal[:, 0], cmap='viridis',
#                      edgecolors='black', linewidth=0.5)
# ax.set_xlabel(f'PC2 ({explained_variance_ratio[1]*100:.2f}% var)', fontsize=12)
# ax.set_ylabel(f'PC3 ({explained_variance_ratio[2]*100:.2f}% var)', fontsize=12)
# ax.set_title('PC2 vs PC3: Colored by PC1 values', fontsize=14, fontweight='bold')
# ax.grid(True, alpha=0.3)
# ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
# ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
# cbar = plt.colorbar(scatter, ax=ax)
# cbar.set_label('PC1 Score', fontsize=10)
# plt.tight_layout()
# plt.show()

# print("\nCOMMENTARY:")
# print("- This view removes PC1's dominant influence to examine PC2-PC3 relationship")
# print("- More uniform distribution suggests these components capture independent aspects")
# print("- Color gradient (PC1) shows outlier players tend to have extreme PC1 scores")
# print("- Central clustering indicates most players are similar in PC2-PC3 space")
# print("=" * 60)
# print()